# Catch the Eavesdropper (BB84)

*Secure the Grid, step 1 · UTC Problem 3*

Share a secret key over a simulated fiber link, then add an eavesdropper and catch them by the errors they cause.

**Your job:** fill in the encoding step and the eavesdropper, then report the error rate (QBER) with and without them.

> **Before you run anything, pick the kernel.** When this notebook opens, a **Select Kernel** box appears with *Python 3 (ipykernel)* already selected. That's the wrong one: change it to **qBraid-SDK (v0.12.2)**, then click **Select**. Picked the wrong one, or closed the box? Click the kernel name at the top right of the notebook and choose **qBraid-SDK (v0.12.2)**. The first cell checks, and tells you if it's wrong.

In [1]:
# Use the qBraid-SDK (v0.12.2) kernel. This stops with a clear message if another one is picked.
import importlib.util
missing = [m for m in ("qbraid", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Wrong kernel: this one has no " + " or ".join(missing) + ". Click the kernel name "
                      "at the top right of the notebook, pick qBraid-SDK (v0.12.2), then run this cell again.")

# Qiskit Aer is Qiskit's simulator. It runs right here in your Lab instance: instant, free and noise-free,
# with no job and no credits. The first run installs it (a few seconds), pinned to the version tested here.
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer==0.18.2

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()
rng = np.random.default_rng(42)

n_bits = 200          # number of qubits Alice sends (a few seconds to run)

print("Setup OK: the qBraid kernel and Qiskit Aer are ready. You're good to go.")

Setup OK: the qBraid kernel and Qiskit Aer are ready. You're good to go.


In [7]:
# Alice's random bits and random bases (0 = Z/standard, 1 = X/Hadamard)
alice_bits  = rng.integers(0, 2, n_bits)
alice_bases = rng.integers(0, 2, n_bits)
bob_bases   = rng.integers(0, 2, n_bits)


In [8]:
def encode(bit, basis):
    """Return a 1-qubit circuit preparing Alice's bit in the chosen basis."""
    qc = QuantumCircuit(1, 1)

    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)

    return qc

In [9]:
def measure(qc, basis):
    """Measure a 1-qubit circuit in the given basis; return the classical bit."""
    if basis == 1:
        qc.h(0)                     # rotate X basis back to Z before measuring
    qc.measure(0, 0)
    counts = sim.run(qc, shots=1).result().get_counts()   # Aer runs x, h and measure as is: no transpile, much faster
    return int(max(counts, key=counts.get))


In [10]:
def run_bb84(eavesdrop=False):
    bob_results = []

    for i in range(n_bits):
        qc = encode(alice_bits[i], alice_bases[i])

        if eavesdrop:
            eve_basis = rng.integers(0, 2)
            eve_bit = measure(qc, eve_basis)
            qc = encode(eve_bit, eve_basis)

        bob_results.append(measure(qc, bob_bases[i]))

    return np.array(bob_results)

In [11]:
def qber(bob_results):
    """Keep matching-basis rounds and calculate the error rate."""
    keep = alice_bases == bob_bases
    a = alice_bits[keep]
    b = bob_results[keep]

    if len(a) == 0:
        return 0.0

    return np.mean(a != b)


clean = run_bb84(eavesdrop=False)
print("QBER, no eavesdropper:", round(qber(clean), 3))

eve = run_bb84(eavesdrop=True)
print(
    "QBER, with eavesdropper:",
    round(qber(eve), 3),
    "(expect about 0.25)"
)

QBER, no eavesdropper: 0.0
QBER, with eavesdropper: 0.243 (expect about 0.25)


### What if Eve only intercepts some of the qubits?

Here Eve intercepts each qubit with a chosen probability. We plot the error rate against that probability to see if a careful spy could hide inside normal noise.

In [ ]:
import matplotlib.pyplot as plt


def run_bb84_partial(fraction):
    """Same as run_bb84, but Eve only intercepts each qubit with this probability."""
    bob_results = []

    for i in range(n_bits):
        qc = encode(alice_bits[i], alice_bases[i])

        if rng.random() < fraction:
            eve_basis = rng.integers(0, 2)
            eve_bit = measure(qc, eve_basis)
            qc = encode(eve_bit, eve_basis)

        bob_results.append(measure(qc, bob_bases[i]))

    return np.array(bob_results)


fractions = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
sweep_qber = []

for fraction in fractions:
    result = qber(run_bb84_partial(fraction))
    sweep_qber.append(result)
    print("Eve intercepts", fraction, "-> QBER", round(result, 3))

hardware_qber = 0.004      # from our IonQ Forte run, with no eavesdropper
bb84_limit = 0.11          # above about 11% BB84 cannot make a secure key

plt.figure(figsize=(8, 5))
plt.plot(fractions, sweep_qber, "o-", label="Simulated QBER")
plt.plot(fractions, [f / 4 for f in fractions], "--", color="gray", label="Expected (fraction / 4)")
plt.axhline(hardware_qber, color="green", linestyle=":", label="Hardware noise with no spy (0.004)")
plt.axhline(bb84_limit, color="red", linestyle=":", label="BB84 gives up (about 0.11)")
plt.xlabel("Fraction of qubits Eve intercepts")
plt.ylabel("QBER (error rate)")
plt.title("Error rate when Eve intercepts only some qubits")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Hardware run: one job, not hundreds

The simulator cells above send one qubit per run, which is fine on a free local simulator. On real hardware that would be hundreds of queued, billed jobs. Instead, put 16 BB84 rounds side by side in one circuit (qubit *k* carries round *k*) and run it once with 200 shots.

Every shot repeats the same 16 rounds. That's fine here: Part 3 asks for the error rate caused by hardware noise, and 200 shots × the sifted rounds (6 of the 16 with this seed) give you about 1,200 measured bits to estimate it. They're 200 repeats of the same few states on the same few qubits, so treat the result as a rough estimate.

In [24]:
def bb84_batch_circuit(bits, bases, bob_bases):
    """Put one BB84 round on each qubit in one circuit."""
    k = len(bits)
    qc = QuantumCircuit(k, k)

    for q in range(k):
        has_gate = False

        if bits[q] == 1:
            qc.x(q)
            has_gate = True

        if bases[q] == 1:
            qc.h(q)
            has_gate = True

        if bob_bases[q] == 1:
            qc.h(q)
            has_gate = True

        if not has_gate:
            qc.id(q)

    qc.measure(range(k), range(k))
    return qc


def qber_from_counts(counts, bits, bases, bob_bases):
    """Calculate the error rate for rounds where Alice and Bob used the same basis."""
    keep = [q for q in range(len(bits)) if bases[q] == bob_bases[q]]
    errors = 0
    total = 0

    for bitstring, count in counts.items():
        bob = bitstring.zfill(len(bits))[::-1]

        for q in keep:
            errors += count * (int(bob[q]) != bits[q])
            total += count

    if total == 0:
        return 0.0

    return errors / total


k = 16
hw_args = (alice_bits[:k], alice_bases[:k], bob_bases[:k])
hw_qc = bb84_batch_circuit(*hw_args)

counts = sim.run(transpile(hw_qc, sim), shots=200).result().get_counts()
print("Batched QBER on the local simulator:",
      round(qber_from_counts(counts, *hw_args), 3))

Batched QBER on the local simulator: 0.0


### Run it on a quantum computer through qBraid

Practice first on IonQ's simulator. It's free, and its `noise_model` option adds IonQ's gate noise. That noise sits mostly on two-qubit gates, so a circuit with few of them (like this one) comes back almost noise-free; real hardware adds readout error on top. Quantinuum's free H2 emulator models readout error too: the cell has the lines to try it. When your circuit works, switch the two lines to the real IonQ QPU for your hardware run. That run uses credits, so cap it at 200 shots and copy the job ID into your write-up.

In [ ]:
# --- Practice run on IonQ's free simulator (run this before the hardware cell) ---
from qbraid.runtime import QbraidProvider
import warnings

warnings.filterwarnings("ignore", message="Circuit contains measurement gates")

sim_device = QbraidProvider().get_device("ionq:ionq:sim:simulator")
sim_job = sim_device.run(hw_qc, shots=200, runtime_options={"noise_model": "aria-1"})

print("Job ID:", sim_job.id)

sim_counts = sim_job.result().data.get_counts()
print("QBER on the IonQ simulator:",
      round(qber_from_counts(sim_counts, *hw_args), 3))

In [25]:
# --- Run on a quantum computer through qBraid ---
from qbraid.runtime import QbraidProvider
import warnings

warnings.filterwarnings("ignore", message="Circuit contains measurement gates")

provider = QbraidProvider()
device = provider.get_device("openquantum:ionq:qpu:forte-1")
job = device.run(hw_qc, shots=200)

print("Job ID:", job.id)

hw_counts = job.result().data.get_counts()
print("Most common results:",
      dict(sorted(hw_counts.items(), key=lambda item: -item[1])[:5]))
print("QBER from this hardware run:",
      round(qber_from_counts(hw_counts, *hw_args), 3))

Job ID: openquantum:ionq:qpu:forte-1-b095-qjob-6af88c54c8f5265fb3f1bdde
Most common results: {'1100011010110100': 7, '1100011010100101': 6, '1100011110100100': 6, '1101011000100101': 6, '1101011000110111': 6}
QBER from this hardware run: 0.004


In [27]:
print("Job status:", job.status())

result = job.result()
print("Result success:", result.success)
print("Result details:", result.details)

if result.success:
    hw_counts = result.data.get_counts()
    print("Counts:", hw_counts)
    print("Hardware QBER:",
          round(qber_from_counts(hw_counts, *hw_args), 3))

Job status: JobStatus.COMPLETED
Result success: True
Result details: {'time_stamps': TimeStamps(createdAt=datetime.datetime(2026, 11, 14, 16, 29, 12, 909000, tzinfo=datetime.timezone.utc), endedAt=datetime.datetime(2026, 11, 14, 16, 35, 50, 99954, tzinfo=datetime.timezone.utc), executionDuration=397190), 'cost': Credits('0'), 'status': <COMPLETED: 'job has successfully run'>}
Counts: {'0100011010110101': 1, '1100001010100111': 1, '1100011000100100': 2, '1100011000100101': 4, '1100011000100110': 4, '1100011000100111': 2, '1100011000110100': 3, '1100011000110101': 3, '1100011000110111': 3, '1100011010010110': 1, '1100011010100100': 4, '1100011010100101': 6, '1100011010100111': 2, '1100011010110000': 1, '1100011010110100': 7, '1100011010110101': 3, '1100011010110110': 2, '1100011010110111': 2, '1100011100100100': 2, '1100011100100101': 4, '1100011100100110': 2, '1100011100100111': 2, '1100011100110011': 1, '1100011100110100': 4, '1100011100110101': 1, '1100011100110110': 2, '1100011100110

In [23]:
from qiskit import qasm3

gate_qubits = sorted({
    hw_qc.find_bit(qubit).index
    for item in hw_qc.data
    if item.operation.name not in ("measure", "barrier")
    for qubit in item.qubits
})

print("Declared qubits:", hw_qc.num_qubits)
print("Qubits used by gates:", gate_qubits)
print("OpenQASM sent for compilation:")
print(qasm3.dumps(hw_qc))

Declared qubits: 16
Qubits used by gates: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15]
OpenQASM sent for compilation:
OPENQASM 3.0;
include "stdgates.inc";
bit[16] c;
qubit[16] q;
x q[0];
h q[0];
x q[1];
h q[1];
x q[2];
h q[2];
h q[2];
h q[3];
h q[3];
x q[4];
h q[4];
x q[5];
h q[5];
h q[5];
h q[6];
h q[6];
x q[7];
h q[7];
x q[8];
h q[8];
x q[9];
h q[9];
h q[9];
x q[10];
h q[11];
h q[11];
x q[12];
h q[12];
x q[14];
h q[14];
h q[14];
x q[15];
h q[15];
h q[15];
c[0] = measure q[0];
c[1] = measure q[1];
c[2] = measure q[2];
c[3] = measure q[3];
c[4] = measure q[4];
c[5] = measure q[5];
c[6] = measure q[6];
c[7] = measure q[7];
c[8] = measure q[8];
c[9] = measure q[9];
c[10] = measure q[10];
c[11] = measure q[11];
c[12] = measure q[12];
c[13] = measure q[13];
c[14] = measure q[14];
c[15] = measure q[15];



## What to have ready
Write your answers in `WRITEUP.md` in this folder (see "Due Sunday" in the track README).
- **Part 1:** near-zero QBER with no eavesdropper (your sifted keys match).
- **Part 2:** the QBER with and without the eavesdropper.
- **Part 3:** your hardware QBER (and the Quantinuum emulator's, if you ran it), your threshold reasoning, and the job IDs.
- **Part 4:** the written physics explanation.
- **For your deck:** the error-rate jump is your evidence that physics can catch a spy. What does it mean for the utility?
